<div style='float: right'><img src='pic/snake_puzzle.png'/></div>

## <div id='snake_puzzle' />スネークパズル

In [ ]:
from itertools import pairwise
import numpy as np
from mip import Model, xsum

h_wall = [(1,0)]  # 横の太線
v_wall = []  # 縦の太線
data = np.array([
    [4,1,0,0],
    [0,0,2,0],
    [0,3,0,0],
    [0,0,0,0],
])

### 問題
* 1から4までを順に線でつなぐ
* すべてのマスを通ること
* 太線は通らないこと

### 変数
* vo：順番
* vl：左に行くか
* vt：上に行くか
* vr：右に行くか
* vb：下に行くか

### 制約
* マスに入る線は1つ（開始以外）
* マスから出る線は1つ（終了以外）
* vl, vt, vr, vbの制約条件
* 数字の順に通ること
* 太線を通らないこと

In [ ]:
n = len(data)
mx = n*n
mx_data = data.max()
o2yx = {int(data[y,x]):(y,x) for y in range(n) for x in range(n) if data[y,x]}

m = Model()
vo = m.add_var_tensor((n, n), "o", ub=mx-1)
vl = m.add_var_tensor((n, n-1), "l", var_type="B")
vt = m.add_var_tensor((n-1, n), "t", var_type="B")
vr = m.add_var_tensor((n, n-1), "r", var_type="B")
vb = m.add_var_tensor((n-1, n), "b", var_type="B")

def v_in(y, x):
    if x < n-1: yield vl[y, x]
    if y < n-1: yield vt[y, x]
    if x: yield vr[y,x-1]
    if y: yield vb[y-1,x]

def v_out(y, x):
    if x: yield vl[y,x-1]
    if y: yield vt[y-1,x]
    if x < n-1: yield vr[y, x]
    if y < n-1: yield vb[y, x]

for y in range(n):
    for x in range(n):
        m += xsum(v_in(y, x)) == ((y,x) != o2yx[1])
        m += xsum(v_out(y, x)) == ((y,x) != o2yx[mx_data])
for y in range(n):
    for x in range(n-1):
        m += vo[y, x]+1 <= vo[y,x+1] + (1-vr[y,x])*mx
        m += vo[y, x+1]+1 <= vo[y,x] + (1-vl[y,x])*mx
for y in range(n-1):
    for x in range(n):
        m += vo[y, x]+1 <= vo[y+1,x] + (1-vb[y,x])*mx
        m += vo[y+1, x]+1 <= vo[y,x] + (1-vt[y,x])*mx
for o1, o2 in pairwise(range(1,mx_data+1)):
    m += vo[o2yx[o1]]+1 <= vo[o2yx[o2]]
for y,x in h_wall:
    vt[y,x].ub=vb[y,x].ub=0
for y,x in v_wall:
    vl[y,x].ub=vr[y,x].ub=0

m.verbose = 0
m.optimize()
out = [["  "]*(2*n+1) for _ in range(2*n+1)]
for y in range(n):
    for x in range(n):
        if data[y,x]:
            out[2*y+1][2*x+1] = f"{data[y,x]:02}"
        if x < n-1 and vl[y, x].x+vr[y, x].x > 0.5:
            out[2*y+1][2*x+2] = "--"
        if y < n-1 and vt[y, x].x+vb[y, x].x > 0.5:
            out[2*y+2][2*x+1] = " |"
print("\n".join("".join(i) for i in out))